# Case 2 — Cyclic time features: input-only vs. reconstructed

**Reproduces:** Table 4.23 mechanism, Fig 4.15 (contextual anomalies)

Three MLP-VAE variants on contextual anomalies: (1) plain, no time features; (2) sin/cos(hour, day-of-year) appended as input but not reconstructed; (3) the same features also included in the reconstruction target — this is what the thesis calls 'MLP-VAE-Cyclic'. The thesis's key finding: (2) barely helps over (1), but (3) is a large jump — reconstruction pressure is what forces the model to actually keep calendar phase in the latent code.

Runtime menu -> Change runtime type -> GPU, then run all cells.

This notebook runs on the real ERA5 slice shipped with the repo under `data/era5/` (12 years, centered on the same warmup/test split used at full scale — see `DATA_LICENSE.md`) — no download needed. Numbers will still differ from the thesis's full-scale figures (much shorter warmup/test period, noisier), but the *qualitative* effect described above should still show up.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = "https://github.com/hadasecohen/streaming-vae-anomaly-detection.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

## Run the suite

`notebooks/cases/case02_cyclic_features_suite.yaml` layers `modules/era5_common.yaml` with the architecture / anomaly-type / stream-mode / feature-engineering fragments for each of the runs below (see the suite file for the exact overrides). Runs execute sequentially. Window-mode runs are batched and fast (well under a minute each on GPU); point-mode runs stream one gradient step per row with no batching, so they run on CPU instead (faster than GPU for this access pattern) and take a few minutes each — `modules/stream/point.yaml` caps them to a 5,000-row subset for this reason.

In [ ]:
!python run_regression.py notebooks/cases/case02_cyclic_features_suite.yaml \
    --session runs/regression/case02_cyclic_features --gpus 0

## Compare the runs

`cross_compare.py` walks the session directory, extracts F1/AUC/precision/recall from each run's `trial_predictions.csv`, and writes a performance table plus comparison plots (score histograms, F1-vs-variant line plot, confusion grid, seed-stability heatmap) under `<session>/cross_compare/`.

In [ ]:
!python cross_compare.py runs/regression/case02_cyclic_features

In [ ]:
import pandas as pd
perf = pd.read_csv("runs/regression/case02_cyclic_features/cross_compare/performance_table.csv")
perf[["run_name", "arch", "anomaly", "variant", "f1", "auc", "precision", "recall", "tp", "fp", "fn"]]

In [ ]:
# Display the key comparison plot(s) inline
import glob
from IPython.display import Image, display

print("Score histograms (normal vs anomaly) and F1-vs-variant line plot:")
for p in sorted(glob.glob("runs/regression/case02_cyclic_features/cross_compare/contextual/score_hist_MLP.png")):
    display(Image(filename=p))
for p in sorted(glob.glob("runs/regression/case02_cyclic_features/cross_compare/contextual/section_lines_cyclic.png")):
    display(Image(filename=p))